In [2]:
import open_clip
import torch

# Load a small, fast CLIP variant — good for getting started
model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='openai'
)
tokenizer = open_clip.get_tokenizer('ViT-B-32')
model.eval()  # inference mode, not training

C:\Users\prith\anaconda3\envs\torch_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\prith\anaconda3\envs\torch_env\lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-11): 12 x ResidualAttentionBlock(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((768,), eps=1e-05, elementwise_affine

In [7]:
from PIL import Image

image = Image.open("../data/images/dog.jpeg")
image_input = preprocess(image).unsqueeze(0)  # add batch dimension

with torch.no_grad():  # no need to track gradients, we're not training
    image_features = model.encode_image(image_input)

print(image_features.shape)   # should print: torch.Size([1, 512])

torch.Size([1, 512])


In [10]:
text_input = tokenizer(["a photo of a mountain"])

with torch.no_grad():
    text_features = model.encode_text(text_input)

print(text_features.shape)   # should also print: torch.Size([1, 512])

torch.Size([1, 512])


In [11]:
import torch.nn.functional as F

# normalize both vectors, then measure cosine similarity
image_features = F.normalize(image_features, dim=-1)
text_features = F.normalize(text_features, dim=-1)

similarity = (image_features @ text_features.T).item()
print(similarity)

0.19160835444927216
